In [1]:
import os
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [2]:
def extract_features(file_path):
    y, sr = librosa.load(file_path)
    centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=sr), axis=1).mean()
    return [centroid, zcr, rolloff, chroma]

In [3]:
# Directories
running_playlist_folder = 'Blues'
sleeping_playlist_folder = 'NotBlues'

In [4]:
# Extract features, labels, and filenames
features, labels, filenames = [], [], []
for folder, label in [(running_playlist_folder, 0), (sleeping_playlist_folder, 1)]:
    for file in os.listdir(folder):
        if file.endswith('.aiff'):
            path = os.path.join(folder, file)
            features.append(extract_features(path))
            labels.append(label)
            filenames.append(file)  # Store the filename

In [5]:
# Convert to numpy arrays for machine learning
features = np.array(features)
labels = np.array(labels)

In [6]:
# Splitting the dataset into training and testing sets along with filenames
features_train, features_test, labels_train, labels_test, filenames_train, filenames_test = train_test_split(
    features, labels, filenames, test_size=0.25, random_state=42)

In [7]:
# Feature scaling
scaler = StandardScaler()
features_train_scaled = scaler.fit_transform(features_train)
features_test_scaled = scaler.transform(features_test)

In [8]:
# Neural network training
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500)
mlp.fit(features_train_scaled, labels_train)

MLPClassifier(max_iter=500)

In [9]:
# Prediction and evaluation
predicted_labels = mlp.predict(features_test_scaled)
accuracy = accuracy_score(labels_test, predicted_labels)

In [10]:
# Displaying each file name with its predicted label
for filename, true_label, predicted_label in zip(filenames_test, labels_test, predicted_labels):
    print(f"{filename} - True Label: {true_label} - Predicted Label: {predicted_label}")

print(f'\nTest Accuracy: {accuracy}')

The Thrill Is Gone.aiff - True Label: 0 - Predicted Label: 0
It's Too Late.aiff - True Label: 1 - Predicted Label: 0

Test Accuracy: 0.5
